<a href="https://colab.research.google.com/github/romenmeitei/Computational-analysis-of-SMA/blob/main/Sensitivity_analysis_of_the_composite_scoring_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np

# -----------------------------
# Load docking data
# -----------------------------
dock = pd.read_csv("All_Ligands_Merged_Docking.csv")

# Extract PubChem CID
dock["PubChem_CID"] = dock["Ligand"].str.extract(r"^(\d+)").astype(int)

# -----------------------------
# Load PubChem descriptor dataset
# -----------------------------
pubchem = pd.read_csv("PubChem_compound.csv")

# Standardize column names
pubchem.columns = pubchem.columns.str.strip().str.replace(" ", "_")

# Merge datasets
data = dock.merge(
    pubchem,
    left_on="PubChem_CID",
    right_on="Compound_CID",
    how="inner"
)

# -----------------------------
# Compute ranking variables
# -----------------------------

# Absolute docking energy
data["Abs_Energy"] = abs(data["Docking_Energy"])

# Pose stability
epsilon = 1e-3
data["Pose_Stability"] = 1 / (data["RMSD_lb"] + epsilon)

# Rename descriptors if needed
data = data.rename(columns={
    "H-Bond_Donor_Count": "HBD",
    "H-Bond_Acceptor_Count": "HBA"
})

# ADMET descriptor list
admet_cols = [
    "Molecular_Weight",
    "XLogP",
    "Polar_Area",
    "HBD",
    "HBA",
    "Rotatable_Bond_Count"
]

# Z-score normalization
for col in admet_cols:
    data[f"Z_{col}"] = (data[col] - data[col].mean()) / data[col].std()

# ADMET proxy
data["ADMET_proxy"] = data[[f"Z_{c}" for c in admet_cols]].mean(axis=1)

# Normalize docking terms
data["Z_Energy"] = (data["Abs_Energy"] - data["Abs_Energy"].mean()) / data["Abs_Energy"].std()
data["Z_Stability"] = (data["Pose_Stability"] - data["Pose_Stability"].mean()) / data["Pose_Stability"].std()

# -----------------------------
# Sensitivity analysis weights
# -----------------------------

weights = {
    "Base": (0.5,0.2,0.3),
    "Alt_1": (0.6,0.2,0.2),
    "Alt_2": (0.4,0.3,0.3)
}

# Composite scores
for key,(wE,wS,wA) in weights.items():
    data[key] = (
        wE * data["Z_Energy"] +
        wS * data["Z_Stability"] +
        wA * data["ADMET_proxy"]
    )

# Ranking (integer ranks)
for key in weights.keys():
    data[key+"_Rank"] = data[key].rank(ascending=False, method="min")

# -----------------------------
# Ligand-level ranking
# -----------------------------

ranked = data.sort_values("Base", ascending=False)

best_per_ligand = (
    ranked
    .groupby("PubChem_CID", as_index=False)
    .first()
)

# -----------------------------
# Extract top ligands
# -----------------------------

table_s2 = best_per_ligand.sort_values("Base_Rank").head(20)

# Save Table S2
table_s2[[
    "PubChem_CID",
    "Name",
    "Base_Rank",
    "Alt_1_Rank",
    "Alt_2_Rank"
]].to_csv("Table_S2_Sensitivity_Analysis.csv", index=False)

print("Table S2 generated successfully.")

Table S2 generated successfully.
